# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/hieutrungdao/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = "8"         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null


## Hoàn tất nộp bài + Bonus

Các ô dưới đây: (5) auth GitHub qua Colab Secrets — không gõ token vào ô nào, (6) bonus B1 NB6, (7) bonus B4 quét rank, (8) verify + push tất cả.

**Trước ô 5**: mở panel Secrets (icon 🔑 bên trái) → Add new secret → tên `GH_TOKEN`, giá trị là Personal Access Token GitHub (scope `repo`) → bật "Notebook access". Không dán token vào bất kỳ ô code hay chat nào.

In [ ]:
# @title 5. Auth + pull (dùng Colab Secrets, không gõ token vào ô)
from google.colab import userdata
import os
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
REMOTE = "https://huyngo3113:" + os.environ["GH_TOKEN"] + "@github.com/huyngo3113/Day21-Track3-Finetuning-Lab.git"
!git pull {REMOTE} main


In [ ]:
# @title 6. Bonus B1 — NB6 merge + hot-swap (+3, ~10 phút)
import os
os.environ["COMPUTE_TIER"] = "T4"
os.environ.pop("EVAL_LIMIT", None)
!python scripts/colab_run.py nb6


In [ ]:
# @title 7. Bonus B4 — quét rank r ∈ {8, 16, 64} (+3, ~30-35 phút)
# r=16 == `correct` đã train ở ô 3, script chỉ train thêm r=8 và r=64.
!python scripts/rank_sweep.py


In [ ]:
# @title 8. Verify + push tất cả (core + B1 + B4)
!python scripts/verify.py
!git add -f submission/REPORT.md results/*.json results/*.csv adapters/correct
!git status
!git commit -m "NB6 merge/hot-swap + rank sweep (B4) + full qualitative.json + adapters/correct"
!git push {REMOTE} HEAD:main
